# <center> AI 1013 : Programming for AI </center>
## <center> Assignment 2 </center>
### 2. Classification using the $k$-Nearest Neighbors Technique

#### 2.1 The Abalone Dataset

Importing the abalone dataset using the code snippet provided.

In [ ]:
import pandas as pd
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data"
abalone = pd.read_csv(url, header=None)
abalone.head(3)

#### 2.2 Data Pre-Processing and Cleaning

To make the dataset easier to understand and work on, we start by renaming the columns to a more meaningful name.

In [ ]:
column_names = ["Sex", "Length", "Diameter","Height","Whole weight", "Shucked weight", "Viscera weight", "Shell weight", "Rings"]
abalone.columns = column_names
abalone.head(3)

Since Sex of the abalone does not directly influence the age of the abalone, we can remove this parameter from our dataset.

In [ ]:
abalone.drop(columns = "Sex", inplace = True)
abalone.head(3)

#### 2.3 Training and Test Data

We will separate this data into two frames, one which contains all the features and the other which contains the target feature (Rings)

In [ ]:
abalone_features = abalone.drop(columns = "Rings")
abalone_rings = abalone["Rings"]

Now we will be splitting our dataset into two parts, one for training and the other for evaluation.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(abalone_features.values, abalone_rings.values, test_size=0.3, random_state=333)

Now we have our training data and test data separated. <br> The X values are large arrays of 7-D vectors. Each component of the vector is the numerical value of some feature of the abalone. <br> The y values are arrays containing the number of rings each abalone has.<br>We will have that the number of elements in X_train and y_train will be same as each abalone defined by a particular vector (in X_train) has a known number of rings (in y_train).

#### 2.4 Implementing the $k$-Nearest Neighbors Algorithm

We need to predict the age of a new abalone given by following attributes using the $k$-nearest neighbors method.
<table>
    <thead>
        <tr>
            <th>Variable</th>
            <th>Value</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>Length</td>
            <td>$0.569552$</td>
        </tr>
        <tr>
            <td>Diameter</td>
            <td>$0.446407$</td>
        </tr>
        <tr>
            <td>Height</td>
            <td>$0.154437$</td>
        </tr>
        <tr>
            <td>Whole Weight</td>
            <td>$1.016849$</td>
        </tr>
        <tr>
            <td>Shucked Weight</td>
            <td>$0.439051$</td>
        </tr>
        <tr>
            <td>Viscera Weight</td>
            <td>$0.222526$</td>
        </tr>
        <tr>
            <td>Shell weight</td>
            <td>$0.291208$</td>
        </tr>
    </tbody>
</table>

Let's define this table as an array in our code.

In [ ]:
import numpy as np
new_abalone = np.array([0.569552, 0.446407, 0.154437, 1.016849,0.439051,0.222526,0.291208])

Calculating euclidean distance between each row of `X_train` and the `new_abalone`

In [ ]:
distances = []
for training_point in X_train:
    distances.append(np.linalg.norm(training_point - new_abalone))
distances = np.array(distances)

Sorting the `distances` values and finding the indices of the $3$ nearest neighbors of the `new_abalone`

In [ ]:
#getting sorted list of indices (in ascending order)
indices = distances.argsort()

#keeping only the first k elements (here k=3)
indices = indices[:3]
print(f"The 3 nearest neighbors have indices {indices[0]}, {indices[1]} and {indices[2]}.")

#putting the corresponding y_values(age / no. of rings) in an array
k_nearest_ages = np.array([y_train[i] for i in indices])
print(f"The corresponding number of rings are {k_nearest_ages[0]}, {k_nearest_ages[1]} and {k_nearest_ages[2]}.")

Now to calcuate mode as per our requirements, we will define a function.

In [ ]:
from statistics import multimode

def find_mode(numbers):
    #getting an array of mode(s)
    mode = multimode(numbers)

    #Return the mode
    #return randmoly if multiple modes found
    np.random.seed(123)
    return np.random.choice(mode)

Now calculating the mode of the data.

In [ ]:
predicted_age = find_mode(k_nearest_ages)
print(f"The predicted age of the new_abalone is {predicted_age}")

Now, as directed we will predict the age of the abalone datapoints in the `X_test` and calcuate the MSE between predicted value and the actual value of the age. <br>
We will ease our work by defining a function which will predict the age.

In [ ]:
def predict_age(k, X_train, y_train, abalone_point):
    #Calculating euclidean distances
    distances = np.linalg.norm(X_train - abalone_point, axis=1)

    #getting the k nearest neighbours
    indices = np.argsort(distances)
    indices = indices[:k]

    #fetching the corresponding y values in an array
    k_nearest_ages = np.array([y_train[i] for i in indices])

    #returning the mode of the k nearest ages as the predicted age
    return find_mode(k_nearest_ages)

Now using this function to predict the ages and store them in array.

In [ ]:
# predicted_ages_test = np.array([predict_age(3, X_train, y_train, data_point) for data_point in X_test])
predicted_ages_test = []
for data_point in X_test:
    predicted_ages_test.append(predict_age(3,X_train, y_train, data_point))
predicted_ages_test = np.array(predicted_ages_test)

Now lets define a function to calculate the MSE and return it.

In [ ]:
def calculate_MSE(predicted, actual):
    squared_sum = 0

    for i in range(len(predicted)):
        squared_sum += (predicted[i] - actual[i])**2

    return squared_sum/len(predicted)

Calculating the MSE for $k=3$ and printing it.

In [ ]:
MSE = calculate_MSE(predicted_ages_test, y_test)
print(f"The MSE for k=3 is {MSE}")

#### 2.5 Tuning $k$ to Achieve Optimal Performance.

In [ ]:
#generating array of k values
k_values = np.arange(1,51)

#Calculating MSE value for all the k values
mse_values = []
for i in range(len(k_values)):
    predicted_ages_test = np.array([predict_age(k_values[i], X_train, y_train, data_point) for data_point in X_test])
    mse_values.append(calculate_MSE(predicted_ages_test, y_test))

mse_values = np.array(mse_values)

Lets now plot `mse_values` vs `k_values`.

In [ ]:
#finding the optimal k-value for which minimum MSE occurs.
optimal_k = mse_values.argsort()[0] + 1
min_mse = mse_values[optimal_k - 1]
print("The Optimal value of k =", optimal_k)
print("MSE for optimal value of k =",min_mse)

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))

#plotting the mse vs k
plt.plot(k_values, mse_values, "o-", label="MSE vs k-value", color="royalblue")

#plotting the optimal k value
plt.axvline(x=optimal_k, color='r', linestyle='--', label=f'Optimal k = {optimal_k}, MSE = {min_mse:.4f}')

plt.legend()
plt.grid(linestyle="--")
plt.xlabel("k value")
plt.ylabel("Mean Squared Error")
plt.title("MSE vs k for k-NN Classification")
plt.show()

The optimal value of $k$ for which MSE is minimum is $k=17$.<br> <br> For better results, lets implement k-fold cross-validation to find the optimal value of $k$.

#### $K$-Fold Cross-Validation

For $K$-fold cross-validation, we will take the value $K = 5$ so that our test data is $20\%$ of the total data.

In [ ]:
num_folds = 5

np.random.seed(15)

#randomly creating folds
indices = np.random.permutation(len(abalone_features))
folds = np.array_split(indices, num_folds)

#making the folds
feature_folds =[abalone_features.iloc[fold] for fold in folds ]
rings_folds = [abalone_rings.iloc[fold] for fold in folds]

Now we have our folds, lets calculate the mean MSE for $k=1$ to $k=30$ to comapre them.

In [ ]:
#generating array of k-values
k_values = np.arange(1,31)

mean_mse_values = [0 for _ in k_values]
#Calculating mean MSE for all k values using k fold
for k in k_values:
    for i in range(num_folds):
        #getting the testing data
        X_test, y_test = feature_folds[i].to_numpy(), rings_folds[i].to_numpy()

        #getting the training data
        X_train = pd.concat([feature_folds[j] for j in range(num_folds) if j!=i]).to_numpy()
        y_train = pd.concat([rings_folds[j] for j in range(num_folds) if j!=i]).to_numpy()

        #Calculating mse and adding to array
        predicted_ages_test = np.array([predict_age(k, X_train, y_train, data_point) for data_point in X_test])
        mean_mse_values[k-1] += (calculate_MSE(predicted_ages_test, y_test))

mean_mse_values = np.array(mean_mse_values)
mean_mse_values /= num_folds

Now we have our mean mse values for all values of $K$, lets plot it and get the optimal value of $k$

In [ ]:
optimal_k_kfolds = mean_mse_values.argsort()[0] + 1
min_mse_kfolds = mean_mse_values[optimal_k_kfolds - 1]
print("The Optimal value of k =", optimal_k_kfolds)
print("MSE for optimal value of k =",min_mse_kfolds)

plt.figure(figsize=(10, 6))

#plotting the mse vs k
plt.plot(k_values, mean_mse_values, "o-", label="MSE vs k-value (Via K-Folds method)", color="royalblue")

#plotting the optimal k value
plt.axvline(x=optimal_k_kfolds, color='r', linestyle='--', label=f'Optimal k = {optimal_k_kfolds}, MSE = {min_mse_kfolds:.4f}')

plt.legend()
plt.grid(linestyle="--")
plt.xlabel("k value")
plt.ylabel("Mean of MSE over multiple folds")
plt.title("MSE vs k for k-NN Classification (Using K-Folds)")
plt.show()

It is clear from the graph that $k=11$ is a better choice for $k$ as compared to $k=17$ obtained by a single test-train split